# Testing and debugging the dataset creation

In [117]:
from dotenv import load_dotenv
import requests
import os
from firecrawl import Firecrawl
import json
from datetime import datetime
import re
from urllib.parse import urlsplit
import time
import sys
sys.path.append("..")
import importlib
import scripts.ingest

importlib.reload(scripts.ingest)

from scripts.ingest import (
    processed_date,
    scrape_archive_metadata,
    selection_logic,
    extract_slug,
    scrape_or_load_edition,
    filter_and_clean_parts,
    clean_q4_and_answer,
    url_headline_extract,
    process_edition,
    append_and_store_edition,
    append_and_store_quiz,
    check_existence_process_append_store
)

load_dotenv("../.env")

api_key = os.getenv("FIRECRAWL_API_KEY")

client = Firecrawl(api_key=api_key)

data_path = "../data"

with open(f"{data_path}/archive.json", "r") as f:
    archive = json.load(f)

In [63]:
full_archive_metadata = scrape_archive_metadata(client)

In [121]:
# Constants and initial steps (just testing on one edition)

first_thursday = datetime.fromisoformat("2020-11-05").date()
first_worded_slug = datetime.fromisoformat("2021-09-30").date()
valid_weekdays = [3,5] # Thursday, Saturday

data_path = "../data"

index = 2366
edition_metadata = full_archive_metadata[index]

id = edition_metadata["_id"]
date = processed_date(edition_metadata)
print(date)
edition_metadata

2019-10-19


{'_id': 'qmHmUiztWr9AzjtgMapfktat',
 'date': '2019-10-19T10:00:00.000+00:00',
 'previewText': 'Why your apartment in Nashville is getting smaller...',
 'slug': '2019/10/19',
 'subjectLine': 'Message received',
 'title': 'Boeing Has Read Receipts On',
 'websitePreviewText': None}

In [122]:
selection_logic(date)

True

In [123]:
slug = extract_slug(edition_metadata, date)
print(slug)

2019%2F10%2F19


In [124]:
edition_metadata = full_archive_metadata[index]
edition_id = edition_metadata["_id"]
iso_date = date.isoformat()

scraped_edition = scrape_or_load_edition(client, slug, edition_id, iso_date, archive)
scraped_edition

Edition for 2019-10-19 found


'[Skip to main content](https://www.morningbrew.com/issues/2019%2F10%2F19#main-content)\n\nWelcome! We use third-party tools to collect information about you as you enjoy the site. Our [Privacy Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) explains more. To opt-out of targeted ads, choose [Do Not Sell / Share My Info](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb). By proceeding, you accept our [Terms](https://www.morningbrew.com/terms-service?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) and the [Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb).\n\nGot it\n\n###### ☕Message received\n\nTo:Brew Readers\n\n* * *\n\n|     |\n| --- |\n| Why your apartment in Nashville is getting smaller... |\n\n{beacon}\n\n| | |     |     |\n| --- | --- |\n| October 19, 2019 | [View Online](https://www.mornin

In [126]:
split_quiz = scraped_edition.split("<br>")
split_quiz

['[Skip to main content](https://www.morningbrew.com/issues/2019%2F10%2F19#main-content)\n\nWelcome! We use third-party tools to collect information about you as you enjoy the site. Our [Privacy Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) explains more. To opt-out of targeted ads, choose [Do Not Sell / Share My Info](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb). By proceeding, you accept our [Terms](https://www.morningbrew.com/terms-service?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) and the [Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb).\n\nGot it\n\n###### ☕Message received\n\nTo:Brew Readers\n\n* * *\n\n|     |\n| --- |\n| Why your apartment in Nashville is getting smaller... |\n\n{beacon}\n\n| | |     |     |\n| --- | --- |\n| October 19, 2019 | [View Online](https://www.morni

In [125]:
questions_and_answers = filter_and_clean_parts(scraped_edition)
questions_and_answers

IndexError: list index out of range

In [72]:
questions_and_answers = clean_q4_and_answer(questions_and_answers)
questions_and_answers

IndexError: list index out of range

In [ ]:
# quiz = {
#         "_id": edition_id,
#         "date": date.isoformat(),
#         "source_url": f"https://www.morningbrew.com/issues/{slug}",
#         "headlines": [],
#         "answer": questions_and_answers[4]
#     }

for index, item in enumerate(questions_and_answers):
    if item[0].isalpha():
        continue
    else:
        headline, url = url_headline_extract(index, questions_and_answers)
        print(headline, url)#quiz["headlines"].append({"text": headline, "url": url})

# quiz

In [120]:
process_edition(index, full_archive_metadata, date, archive, iso_date)

Edition for 2020-10-10 found
Extracting headline text and URLs
{
  "_id": "ADdReyh26gxwpwsbRoVyoQHb",
  "date": "2020-10-10",
  "source_url": "https://www.morningbrew.com/issues/2020%2F10%2F10",
  "headlines": [
    {
      "text": "Julius Caesar did not eat a single root vegetable during his entire life, new study finds",
      "url": "https://media.giphy.com/media/3IpHeyAdAmjQI/giphy.gif?utm_source=morning_brew"
    },
    {
      "text": "Chris Hemsworth helps reintroduce Tasmanian devils to Australia for first time in 3,000 years",
      "url": "https://www.cnet.com/news/chris-hemsworth-helps-reintroduce-tasmanian-devils-to-australia-for-first-time-in-3000-years/#:~:text=Marvel%20actor%20Chris%20Hemsworth%20joined,back%20Tasmanian%20devils%20to%20Australia.?utm_source=morning_brew"
    },
    {
      "text": "T. Rex skeleton brings $31.8 million at Christie\u2019s auction",
      "url": "https://www.nytimes.com/2020/10/06/arts/design/t-rex-skeleton-brings-31-8-million-at-christies-

{'_id': 'ADdReyh26gxwpwsbRoVyoQHb',
 'date': '2020-10-10',
 'source_url': 'https://www.morningbrew.com/issues/2020%2F10%2F10',
 'headlines': [{'text': 'Julius Caesar did not eat a single root vegetable during his entire life, new study finds',
   'url': 'https://media.giphy.com/media/3IpHeyAdAmjQI/giphy.gif?utm_source=morning_brew'},
  {'text': 'Chris Hemsworth helps reintroduce Tasmanian devils to Australia for first time in 3,000 years',
   'url': 'https://www.cnet.com/news/chris-hemsworth-helps-reintroduce-tasmanian-devils-to-australia-for-first-time-in-3000-years/#:~:text=Marvel%20actor%20Chris%20Hemsworth%20joined,back%20Tasmanian%20devils%20to%20Australia.?utm_source=morning_brew'},
  {'text': 'T. Rex skeleton brings $31.8 million at Christie’s auction',
   'url': 'https://www.nytimes.com/2020/10/06/arts/design/t-rex-skeleton-brings-31-8-million-at-christies-auction.html?utm_source=morning_brew'},
  {'text': 'Archaeologists in Egypt crack open 2,500-year-old mummy coffin, video g

In [16]:
q_search = re.compile(r'[1234]\. (“)?.*\[([\w\s\.\-])*\](\(.*\))')
part = '# [Demon Stores Look for New Hosts](https://www.morningbrew.com/daily/stories/2020/10/30/halloween-popup-shops-mixed-success-2020?email={{profile.email}}) |\n\n![Spirit Halloween store](https://dlp31coh2a67q.cloudfront.net/eyJrZXkiOiJ1cGxvYWRzL21lZGl1bS9hc3NldC8yOTkwL3NwaXJpdF9oYWxsb3dlZW5fc3RvcmUuanBnIiwiYnVja2V0Ijoib3Nsby1wcm9kdWN0aW9uIiwiZWRpdHMiOnsicmVzaXplIjp7IndpZHRoIjoxMDAwLCJoZWlnaHQiOm51bGx9fX0=)\n\n_Ben McCanna/Portland Portland Press Herald via Getty Images_\n\n|     |\n| --- |\n| Every year, pop-up Halloween stores rise from the underworld to possess empty storefronts vacated by mortal retailers.\n\nTypically, this non-traditional leasing model means seasonal pop-ups perform well when the rest of retail flounders. But even with 2020 laying out a feast of fresh retail corpses, are spooky stores immune to the pandemic? Let\'s check in with two retail reanimaters:\n\n1. **Spirit Halloween** is the host with the most. It actually opened [more stores](https://www.nytimes.com/2020/10/12/style/spirit-halloween-costume-store-undead-forever.html?utm_source=morning_brew) this year (1,400+) than last year, and its CEO said last month the company is "anticipating a Halloween on par with last year."\n2. **Party City** was far less bullish, opening 91% fewer of its Halloween City locations this year. With just 25 stores across the U.S., these days it\'s more like a Halloween Town.\n\n**Zoom out:** Halloween City was showing zombie-like symptoms even before the pandemic. Last year, sales per store declined nearly [21% annually](https://www.retaildive.com/news/party-citys-october-sales-a-disaster-with-halloween-stores-down-21/566955/?utm_source=morning_brew)—a major problem for parent Party City considering Halloween sales account for a fifth of its total U.S. retail revenue.\n\n|     |\n| --- |\n| [![](https://media.sailthru.com/5z8/1k4/4/2/5e865734bf6f7.jpg)](https://www.facebook.com/sharer/sharer.php?u=https%3A%2F%2Fwww.morningbrew.com%2Fdaily%2Fstories%2F2020%2F10%2F30%2Fhalloween-popup-shops-mixed-success-2020)[![](https://media.sailthru.com/5z8/1k4/4/2/5e86574d1eb82.jpg)](https://twitter.com/intent/tweet?text=Party+City%27s+opening+91%25+fewer+of+its+Halloween+City+locations+this+year&url=https://www.morningbrew.com/daily/stories/2020/10/30/halloween-popup-shops-mixed-success-2020&via=MorningBrew)[![](https://media.sailthru.com/5z8/1k4/4/2/5e86576e04837.jpg)](https://www.linkedin.com/shareArticle?mini=true&url=https%3A%2F%2Fwww.morningbrew.com%2Fdaily%2Fstories%2F2020%2F10%2F30%2Fhalloween-popup-shops-mixed-success-2020&title=Halloween+Pop-Up+Shops+Have+Mixed+Success+In+2020&source=www.morningbrew.com&summary=Party+City%27s+opening+91%25+fewer+of+its+Halloween+City+locations+this+year)[![](https://media.sailthru.com/5z8/1k4/4/2/5e8657887c532.jpg)](mailto:?subject=Check%20out%20this%20story%20from%20Morning%20Brew%21&body=Halloween%20Pop-Up%20Shops%20Have%20Mixed%20Success%20In%202020%3A%20https%3A%2F%2Fwww.morningbrew.com%2Fdaily%2Fstories%2F2020%2F10%2F30%2Fhalloween-popup-shops-mixed-success-2020%0A%0AWant%20more%20great%20content%3F%20Subscribe%20to%20Daily%20Brew%20%E2%80%94%20Delivering%20the%20latest%20business%20news%20from%20Wall%20St.%20to%20Silicon%20Valley%2C%20daily.%0Ahttps%3A%2F%2Fwww.morningbrew.com%2Fdaily%2Fr%2F%3Fkid%3D{{profile.vars.referral_code}}%26utm_source%3Demail_share%0A) | | |\n| --- |\n\n| |     |\n| --- |\n| ### FOOD'
matched = q_search.search(part)
matched.group()

'1. **Spirit Halloween** is the host with the most. It actually opened [more stores](https://www.nytimes.com/2020/10/12/style/spirit-halloween-costume-store-undead-forever.html?utm_source=morning_brew) this year (1,400+)'

In [127]:
with open(f"{data_path}/quizzes.json", "r") as f:
    quizzes = json.load(f)
print(f"Number of quizzes: {len(quizzes)}")

with open(f"{data_path}/exceptions.json", "r") as f:
    exceptions = json.load(f)
print(f"Number of exceptions: {len(exceptions)}")


Number of quizzes: 335
Number of exceptions: 2
